# Data Curation

Scrub our dataset and curate our data

The dataset is here:  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

And the folder with all the product datasets is here:  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories

Initially, we will be using the Clothing, Shoes and Jewelry dataset, streaming the first 20k usable rows, and then 1000 rows for validation and testing.

In [ ]:
# imports

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset, DatasetDict, load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random

from pricer.items import Item
from pricer.jsonl_records import amazon_row_to_record
from pricer.preprocessor import prune_description

load_dotenv(override=True)

True

In [2]:
# Log in to HuggingFace - if you get a "Note" about Environment variable being set, ignore it

hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Load our dataset

In the next cell, we load in the dataset from huggingface.

If this gives you an error like "trust_remote_code is no longer supported", then please run this command in a new cell: `!uv add --upgrade datasets==3.6.0` and then restart the Kernel, and try again.

In [11]:
# Stream category meta, validate rows, prune descriptions, split train/val/test, write JSONL.

DATA_DIR = Path("data") if Path("pricer").is_dir() else Path("wip/babyGPTPricer/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CURATED_JSONL_PATH = DATA_DIR / "clothing_shoes_jewelry_20k.jsonl"
TRAIN_JSONL_PATH = DATA_DIR / "train.jsonl"
VAL_JSONL_PATH = DATA_DIR / "val.jsonl"
TEST_JSONL_PATH = DATA_DIR / "test.jsonl"

TARGET_USABLE_ROWS = 20_000
TRAIN_SIZE = 16_000
VAL_SIZE = 2_000
TEST_SIZE = 2_000
SPLIT_SEED = 42

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_Clothing_Shoes_and_Jewelry",
    split="full",
    streaming=True,
    trust_remote_code=True,
)
# Card schema vs Parquet mismatch on nested `images` (list<struct> vs struct) breaks Arrow cast.
# Project to the columns we use so streaming never loads/casts those fields.
dataset = dataset.select_columns(["title", "description", "features", "price"])


def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as out:
        for row in rows:
            out.write(json.dumps(row, ensure_ascii=False) + "\n")


records = []
written = 0
scanned = 0
with CURATED_JSONL_PATH.open("w", encoding="utf-8") as out:
    for row in tqdm(dataset, desc="Curating", unit="row"):
        scanned += 1
        record = amazon_row_to_record(row)
        if record is None:
            continue

        record["description"] = prune_description(
            record["description"], max_chars=500, target_chars=450
        )
        if len(record["description"]) < 10:
            continue

        out.write(json.dumps(record, ensure_ascii=False) + "\n")
        records.append(record)

        written += 1
        if written >= TARGET_USABLE_ROWS:
            break

if len(records) != TARGET_USABLE_ROWS:
    raise ValueError(f"Expected {TARGET_USABLE_ROWS:,} rows, got {len(records):,}")

rng = random.Random(SPLIT_SEED)
rng.shuffle(records)

train_records = records[:TRAIN_SIZE]
val_records = records[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]
test_records = records[TRAIN_SIZE + VAL_SIZE : TRAIN_SIZE + VAL_SIZE + TEST_SIZE]

if not (
    len(train_records) == TRAIN_SIZE
    and len(val_records) == VAL_SIZE
    and len(test_records) == TEST_SIZE
):
    raise ValueError("Split sizes are incorrect")

write_jsonl(TRAIN_JSONL_PATH, train_records)
write_jsonl(VAL_JSONL_PATH, val_records)
write_jsonl(TEST_JSONL_PATH, test_records)

print(
    f"Wrote curated dataset to {CURATED_JSONL_PATH.resolve()} (rows={written:,}, scanned={scanned:,})"
)
print(
    f"Wrote train/val/test: {TRAIN_JSONL_PATH.name}={len(train_records):,}, "
    f"{VAL_JSONL_PATH.name}={len(val_records):,}, {TEST_JSONL_PATH.name}={len(test_records):,}"
)

curated_jsonl_path = CURATED_JSONL_PATH
curated_rows_written = written
train_jsonl_path = TRAIN_JSONL_PATH
val_jsonl_path = VAL_JSONL_PATH
test_jsonl_path = TEST_JSONL_PATH

Curating: 0row [00:00, ?row/s]

Wrote curated dataset to C:\Dev\Learning\llm_engineering\llm_engineering-ez\wip\babyGPTPricer\data\clothing_shoes_jewelry_20k.jsonl (rows=20,000, scanned=37,404)
Wrote train/val/test: train.jsonl=16,000, val.jsonl=2,000, test.jsonl=2,000


In [ ]:
# Split stats + quick description-length sanity check

print(f"Curated rows written: {curated_rows_written:,} -> {curated_jsonl_path}")
print(f"Train/Val/Test: {train_jsonl_path}, {val_jsonl_path}, {test_jsonl_path}")

sample_lengths = [len(r["description"]) for r in train_records[:200]]
print(
    f"Sample train description lengths (n={len(sample_lengths)}): "
    f"min={min(sample_lengths)}, max={max(sample_lengths)}, avg={sum(sample_lengths)/len(sample_lengths):.1f}"
)

Curated rows written: 20,000 -> data\clothing_shoes_jewelry_20k.jsonl
Train/Val/Test: data\train.jsonl, data\val.jsonl, data\test.jsonl
Sample train description lengths (n=200): min=21, max=500, avg=349.1


In [ ]:
# Sample a curated train JSONL row

sample_train = None
with train_jsonl_path.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 6:
            sample_train = json.loads(line)
            break

print(f"Description chars: {len(sample_train['description'])}")
sample_train

Description chars: 445


{'title': 'Carty Stainless Steel Mesh Watch Band for Men Women,Adjustable Watch Strap 20mm 12mm 14mm 16mm 18mm 24mm 22mm Quick Release Metal Mesh Watch Strap Solid Thin Metal Watch Band with Double Folding Clasp',
 'description': '▶Quality Mesh Bracelets Stainless Steel Mesh Watch Strap 18mm :Soft 304 stainless steel silver watch band with quick release spring bars ▶Size Fit:18mm silver tone stainless steel mesh watch band quick release strap is very comfortable & would fits wrists up to 5.9"-7.87" wrist circumference Mesh bands adopt 0.6mm dia stainless steel wire woven, it\'s stylish and breathable thin solid mesh stainless steel watch band strap replacement bracelet',
 'price': 15.99,
 'bucket': '10-25'}

In [6]:
from collections import Counter

Counter(row["bucket"] for row in train_records)

Counter({'10-25': 6542,
         '25-50': 5752,
         '50-100': 1848,
         '0-10': 1035,
         '100-200': 643,
         '200-500': 167,
         '500+': 13})

In [ ]:
import json
import random
from pathlib import Path

path = Path("data/test.jsonl")
with path.open("r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

for r in random.sample(rows, 5):
    print(r)

{'title': "Teva Women's Omnium Sandal", 'description': "Teva's Omnium hiking shoe brings the best of both worlds This part-sandal, part-shoe hybrid has the light weight and breathability of a sandal, along with the added foot protection of a shoe It also features three points of adjustability for enhanced fit and comfort Teva is an authentic icon in the outdoor industry Founded in the early 1980's by a Colorado River guide, Teva pioneered the sport-sandal category", 'price': 90.0, 'bucket': '50-100'}
{'title': "mysoft Women's Sparkly Dressy Heel Sandals Strappy Wedding Shoes Open Toe Glitter Pump Stilettos", 'description': 'Rubber sole 【Measurement】Heel height measures approximately 3" 【Comfortable & Safe Material】Soft and gentle PU insole provide you a cozy feeling that fits the skin and relieves fatigue Anti-slippery & wear-resistance TPR rubber out-sole ensures steady walking 【Occasions】These perfect heel sandals come in a variety of styles, that can keep you shine at every occasion

In [ ]:
username = "Hatshe"

dataset_name = f"{username}/items_raw_lite"

train_items = [
    Item(
        title=r["title"],
        category="Clothing_Shoes_and_Jewelry",
        price=r["price"],
        full=r["description"],
    )
    for r in train_records
]
val_items = [
    Item(
        title=r["title"],
        category="Clothing_Shoes_and_Jewelry",
        price=r["price"],
        full=r["description"],
    )
    for r in val_records
]
test_items = [
    Item(
        title=r["title"],
        category="Clothing_Shoes_and_Jewelry",
        price=r["price"],
        full=r["description"],
    )
    for r in test_records
]

Item.push_to_hub(dataset_name, train_items, val_items, test_items)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Matplotlib colour charts:

https://matplotlib.org/stable/gallery/color/named_colors.html
